In [2]:
import pandas as pd
import numpy as np
import seaborn as sns

---

## Level 1 — Planets

Use `sns.load_dataset('planets')`. Group by `method` (`observed=True`).

Build a named-agg summary with `n` (count of `number`), `avg_distance` (mean of `distance`), `avg_period` (mean of `orbital_period`), and `period_spread` (max minus min of `orbital_period`).

- Which detection method has the widest spread in orbital periods?
- Some methods have all-NaN distance or period values — drop those rows from the summary first, then check: does average distance correlate with average orbital period across methods?
- Print each method and its planet count in the format `"1. Radial Velocity: 553"`. Use `enumerate`.

In [37]:
planets = sns.load_dataset('planets')
#.dropna(subset=['distance','orbital_period'])

# Your code here

g = planets.groupby('method', observed = True).agg(
    n = ('number','count'),
    avg_distance = ('distance','mean'),
    avg_period = ('orbital_period','mean'),
    period_spread = pd.NamedAgg(column='orbital_period',aggfunc=(lambda x: x.max() - x.min()))
    
)

print(g['period_spread'].idxmax(),'has the widest spread in orbital periods')

gr = g.dropna()

np.corrcoef(gr['avg_distance'], gr['avg_period'])[0,1]
print('slight negative correlation')

for i, (method, count) in enumerate(zip(g.index, g['n']), start=1):
    print(f"{i}. {method}: {count}")


for i, (method, count) in enumerate(zip(g.index, g['n']), start = 1):
    print(f'{i}.{method}: {count}')

Imaging has the widest spread in orbital periods
slight negative correlation
1. Astrometry: 2
2. Eclipse Timing Variations: 9
3. Imaging: 38
4. Microlensing: 23
5. Orbital Brightness Modulation: 3
6. Pulsar Timing: 5
7. Pulsation Timing Variations: 1
8. Radial Velocity: 553
9. Transit: 397
10. Transit Timing Variations: 4
1.Astrometry: 2
2.Eclipse Timing Variations: 9
3.Imaging: 38
4.Microlensing: 23
5.Orbital Brightness Modulation: 3
6.Pulsar Timing: 5
7.Pulsation Timing Variations: 1
8.Radial Velocity: 553
9.Transit: 397
10.Transit Timing Variations: 4


In [39]:
enumerate(zip(g.index, g['n']))

---

## Level 2 — Hotel bookings

`bookings` records hotel stays across two hotel types and three room categories.

Group by `hotel` and `room_type`. Use named agg to compute `total_bookings` (count), `avg_nights` (mean of `nights`), `avg_rate` (mean of `daily_rate`), and `cancel_rate` (mean of `is_cancelled`). Then add an `est_revenue` column: `avg_nights * avg_rate * total_bookings * (1 - cancel_rate)`.

Which (`hotel`, `room_type`) pair generates the most estimated revenue? Which has the highest cancellation rate? Use `np.percentile` to find the median and 90th percentile of `est_revenue` across all groups.

In [24]:
np.random.seed(11)
hotels = ['City', 'Resort']
rooms  = ['Standard', 'Deluxe', 'Suite']
m = 300
bookings = pd.DataFrame({
    'hotel':        np.random.choice(hotels, m),
    'room_type':    np.random.choice(rooms, m),
    'nights':       np.random.randint(1, 8, m),
    'daily_rate':   np.random.choice([89, 129, 159, 199, 249, 349], m).astype(float),
    'is_cancelled': np.random.choice([0, 0, 0, 1], m),
})

# Your code here

g = bookings.groupby(['hotel','room_type']).agg(
    total_bookings = ('nights','count'),
    avg_nights = ('nights','mean'),
    avg_rate =('daily_rate','mean'),
    cancel_rate = ('is_cancelled','mean')
)

g['est_revenue'] = g['avg_nights']*g['avg_rate']*g['total_bookings']*(1-g['cancel_rate'])

print(g['est_revenue'].idxmax(),'has the most estimated revenue')
print(g['cancel_rate'].idxmax(),'has the highest cancellation rate')
print(np.percentile(g['est_revenue'], q = [50,90]))

('City', 'Suite') has the most estimated revenue
('City', 'Standard') has the highest cancellation rate
[30163.962      35152.93642901]


---

## Level 3 — Taxis

Use `sns.load_dataset('taxis').dropna()`. Before grouping, add `tip_pct = tip / fare` (filter to `fare > 0` first).

Group by `payment` and `color`. Build a named-agg summary with `avg_tip_pct` (mean of `tip_pct`), `avg_fare` (mean of `fare`), and `n`. No hints for the questions below.

- Which (`payment`, `color`) group has the highest average tip percentage?
- Across groups, does higher average fare go with higher tip percentage? Print the correlation.
- Which payment type shows the most consistent `avg_tip_pct` across colors? Compute this without a loop.

In [35]:
taxis = sns.load_dataset('taxis').dropna()
taxis = taxis[taxis['fare'] > 0]

# Your code here

taxis['tip_pct'] = taxis['tip'] / taxis['fare']

gg = taxis.groupby(['payment','color']).agg(
    avg_tip_pct = ('tip_pct','mean'),
    avg_fare = ('fare','mean'),
    n = ('fare','count')
)

print(gg['avg_tip_pct'].idxmax(),'has the highest average tip percentage')

print(np.corrcoef(gg['avg_fare'],gg['avg_tip_pct'])[0,1])

ggus = gg['avg_tip_pct'].unstack()
print(ggus.std(axis=1).idxmin(), 'is the most consistent')


('credit card', 'yellow') has the highest average tip percentage
0.5115350787978012
cash is the most consistent
